# ✈️ ETL Pipeline — Flight Delay & Airports Dataset
**FINAL VERSION** — Logging manual (file-flush atomik) sesuai panduan UAS

**Sumber Data:**
- Source 1: `flights_sample_3m.csv` — data keterlambatan penerbangan
- Source 2: `airports.csv` — data bandara global

**Warehouse:** Google BigQuery + SQLite backup (Star Schema)

**Logging:** `write_custom_log()` — open/flush atomik, bukan `logging.basicConfig`

## ⚙️ CELL 1 — Install & Setup

In [ ]:
!pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com -q
!pip install sqlalchemy psycopg2-binary scikit-learn pandas-gbq google-cloud-bigquery -q

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout)
print('✅ Setup selesai')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 51.9 MB/s eta 0:00:00
Sun Jun 14 08:38:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                    

In [ ]:
import cudf
import pandas as pd
import numpy as np
import os, time, subprocess
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sqlalchemy import create_engine, text

print(f'✅ cuDF    : {cudf.__version__}')
print(f'✅ Pandas  : {pd.__version__}')

gpu_info = subprocess.run(
    ['nvidia-smi','--query-gpu=memory.total,memory.free','--format=csv,noheader'],
    capture_output=True, text=True)
print(f'✅ GPU RAM : {gpu_info.stdout.strip()}')

✅ cuDF    : 26.02.01
✅ Pandas  : 2.2.2
✅ GPU RAM : 15360 MiB, 14913 MiB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR  = '/content/drive/MyDrive/TUBES BIG DATA'
RAW_DIR   = os.path.join(BASE_DIR, 'raw')
CLEAN_DIR = os.path.join(BASE_DIR, 'cleaned')
WH_DIR    = os.path.join(BASE_DIR, 'warehouse')

os.makedirs(RAW_DIR,   exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(WH_DIR,    exist_ok=True)

# Batas baris untuk Supabase free tier 500MB
ROW_LIMIT = 150_000

# BigQuery config
GCP_PROJECT = 'tubes-bigdata-498212'   # ← sesuaikan jika perlu
BQ_DATASET  = 'etl_flight_analytics'

print('✅ Folder ETL siap')
print(f'   BASE_DIR  → {BASE_DIR}')
print(f'   RAW_DIR   → {RAW_DIR}')
print(f'   CLEAN_DIR → {CLEAN_DIR}')
print(f'   WH_DIR    → {WH_DIR}')
print(f'   Target    → {ROW_LIMIT:,} baris')

Mounted at /content/drive
✅ Folder ETL siap
   BASE_DIR  → /content/drive/MyDrive/TUBES BIG DATA
   RAW_DIR   → /content/drive/MyDrive/TUBES BIG DATA/raw
   CLEAN_DIR → /content/drive/MyDrive/TUBES BIG DATA/cleaned
   WH_DIR    → /content/drive/MyDrive/TUBES BIG DATA/warehouse
   Target    → 150,000 baris


## 📝 CELL 4 — Modul Logging Manual (File-Flush Atomik)

In [ ]:
# ============================================================
# LOGGING MANUAL — sesuai panduan UAS Genap 25/26
# Menggantikan logging.basicConfig yang sering 0 bytes di Drive
# Menggunakan open() + f.flush() → sync langsung ke FUSE Drive
# ============================================================

log_path = os.path.join(BASE_DIR, 'etl_log.txt')

def write_custom_log(phase, step, status, message, details=None):
    """
    Menulis log terstruktur secara atomik dan memaksa flush ke Google Drive.

    Args:
        phase   : nama fase pipeline (ETL / ELT)
        step    : nama langkah (Extract_Source1, Transform, Load, dll.)
        status  : SUCCESS / FAILED / WARNING
        message : pesan ringkas
        details : dict metadata tambahan (opsional)
    """
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log_entry = (
        f'[{timestamp}] [{phase.upper()}] [{step.upper()}] [{status.upper()}]'
        f' - {message}\n'
    )
    if details:
        for k, v in details.items():
            log_entry += f'   > {k}: {v}\n'

    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(log_entry)
        f.flush()   # ← kunci: paksa sync I/O fisik ke Drive

# Tulis header (overwrite di awal setiap run)
with open(log_path, 'w', encoding='utf-8') as f:
    header = (
        '=== PIPELINE BIG DATA ETL & ELT LOG - UAS GENAP 25/26 ===\n'
        f'Eksekusi dimulai pada: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
        + '=' * 60 + '\n'
    )
    f.write(header)
    f.flush()

print(f'✅ Log ETL → {log_path}')

✅ Log ETL → /content/drive/MyDrive/TUBES BIG DATA/etl_log.txt


## 🔵 EXTRACT
### CELL 5 — ETL Extract Sumber 1 & 2

In [ ]:
# ============================================================
# EXTRACT SOURCE 1 — Flight Delay CSV
# TIDAK ada cleaning/transformasi di sini
# ============================================================

def extract_etl_source1():
    start = time.time()
    path  = os.path.join(BASE_DIR, 'flights_sample_3m.csv')
    ukuran_mb = round(os.path.getsize(path) / 1024**2, 2)

    print(f'   File asli : {ukuran_mb} MB (3 juta baris)')
    print(f'   Mengambil : {ROW_LIMIT:,} baris')

    # Baca dengan cuDF GPU — TIDAK ADA CLEANING
    df = cudf.read_csv(path, nrows=ROW_LIMIT)

    # Simpan ke /raw APA ADANYA
    raw_path = os.path.join(RAW_DIR, 'raw_flights.csv')
    df.to_csv(raw_path, index=False)

    duration = round(time.time() - start, 2)

    write_custom_log('ETL', 'Extract_Source1', 'SUCCESS',
                     'Sukses mengekstrak data penerbangan.',
                     {
                         'Nama Sumber Data'      : 'flights_sample_3m.csv',
                         'Jumlah Baris Extracted': f'{df.shape[0]:,}',
                         'Jumlah Kolom Extracted': df.shape[1],
                         'Ukuran File Asli'      : f'{ukuran_mb} MB',
                         'Waktu Eksekusi'        : f'{duration} detik',
                         'Path Penyimpanan Raw'  : raw_path
                     })

    print(f'\n✅ ETL EXTRACT — Sumber 1: Flight Delay & Cancellation')
    print(f'   Jumlah baris  : {df.shape[0]:,}')
    print(f'   Jumlah kolom  : {df.shape[1]}')
    print(f'   Kolom         : {list(df.columns)}')
    print(f'   Waktu eksekusi: {duration} detik')
    print(f'   Disimpan ke   : {raw_path}')
    return df


# ============================================================
# EXTRACT SOURCE 2 — Airports CSV
# ============================================================

def extract_etl_source2():
    start = time.time()
    path  = os.path.join(BASE_DIR, 'airports.csv')
    ukuran_mb = round(os.path.getsize(path) / 1024**2, 2)

    df = cudf.read_csv(path)

    raw_path = os.path.join(RAW_DIR, 'raw_airports.csv')
    df.to_csv(raw_path, index=False)

    duration = round(time.time() - start, 2)

    write_custom_log('ETL', 'Extract_Source2', 'SUCCESS',
                     'Sukses mengekstrak data bandara.',
                     {
                         'Nama Sumber Data'      : 'airports.csv',
                         'Jumlah Baris Extracted': f'{df.shape[0]:,}',
                         'Jumlah Kolom Extracted': df.shape[1],
                         'Ukuran File Asli'      : f'{ukuran_mb} MB',
                         'Waktu Eksekusi'        : f'{duration} detik',
                         'Path Penyimpanan Raw'  : raw_path
                     })

    print('\n✅ ETL EXTRACT — Sumber 2: Global Airports')
    print(f'   Jumlah baris  : {df.shape[0]:,}')
    print(f'   Jumlah kolom  : {df.shape[1]}')
    print(f'   Kolom         : {list(df.columns)}')
    print(f'   Ukuran file   : {ukuran_mb} MB')
    print(f'   Waktu eksekusi: {duration} detik')
    print(f'   Disimpan ke   : {raw_path}')
    return df


# ── JALANKAN EXTRACT ──
print('=== EXTRACT PHASE ===')
df_flights  = extract_etl_source1()
df_airports = extract_etl_source2()
print('\n🎉 Semua sumber berhasil diekstrak!')

=== EXTRACT PHASE ===
   File asli : 585.69 MB (3 juta baris)
   Mengambil : 150,000 baris

✅ ETL EXTRACT — Sumber 1: Flight Delay & Cancellation
   Jumlah baris  : 150,000
   Jumlah kolom  : 32
   Kolom         : ['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']
   Waktu eksekusi: 3.72 detik
   Disimpan ke   : /content/drive/MyDrive/TUBES BIG DATA/raw/raw_flights.csv

✅ ETL EXTRACT — Sumber 2: Global Airports
   Jumlah baris  : 6,393
   Jumlah kolom  : 13
   Kolom         : ['AirportName', 'IATA', 'ICAO', 'TimeZone', 'City_Name', 'City_IATA', 'UTC_Offset_Hours', 'UTC_Offs

## 🟡 TRANSFORM
### CELL 6 — Data Cleaning

In [ ]:
# ============================================================
# DATA CLEANING
# FIX #1 : fillna dep_delay tidak boleh untuk penerbangan cancelled
# FIX #2 : outlier removal pakai hard cap, BUKAN IQR
# FIX #3 : arr_delay juga perlu outlier handling
# ============================================================
start_time = time.time()

print('=' * 55)
print(' CLEANING — df_flights')
print('=' * 55)
print(f'  Sebelum : {df_flights.shape}')

# Standarkan nama kolom (cuDF)
df_flights.columns = [c.lower().strip() for c in df_flights.columns]

# Hapus duplikat berdasarkan 5 kolom kunci
before_dup = len(df_flights)
df_flights.drop_duplicates(
    subset=['fl_date', 'airline', 'origin', 'dest', 'dep_time'],
    keep='first', inplace=True)
dup_removed = before_dup - len(df_flights)
print(f'  Duplikat dihapus  : {dup_removed:,} baris')

# Hapus baris tanpa kolom kunci
before_null = len(df_flights)
df_flights.dropna(subset=['fl_date', 'airline', 'origin', 'dest'], inplace=True)
print(f'  Null kunci dihapus: {before_null - len(df_flights):,} baris')

# Konversi ke pandas untuk operasi lanjut
df_flights_pd = df_flights.to_pandas()

# Standarkan format tanggal
df_flights_pd['fl_date'] = pd.to_datetime(df_flights_pd['fl_date'], errors='coerce')
before_date = len(df_flights_pd)
df_flights_pd.dropna(subset=['fl_date'], inplace=True)
print(f'  Tanggal invalid   : {before_date - len(df_flights_pd):,} baris')

# Pastikan cancelled bertipe int
if 'cancelled' in df_flights_pd.columns:
    df_flights_pd['cancelled'] = pd.to_numeric(
        df_flights_pd['cancelled'], errors='coerce').fillna(0).astype(int)

# FIX #1 — fillna dep_delay & arr_delay hanya untuk penerbangan TIDAK cancelled
mask_not_cancelled = df_flights_pd['cancelled'] == 0
for col in ['dep_delay', 'arr_delay']:
    if col in df_flights_pd.columns:
        median_val = df_flights_pd.loc[mask_not_cancelled, col].median()
        df_flights_pd.loc[mask_not_cancelled, col] = (
            df_flights_pd.loc[mask_not_cancelled, col].fillna(median_val))
        print(f'  fillna {col}: median={median_val:.1f} (hanya non-cancelled)')

# Tangani missing value kolom lain
for col in ['air_time', 'distance']:
    if col in df_flights_pd.columns:
        df_flights_pd[col] = df_flights_pd[col].fillna(df_flights_pd[col].median())

# FIX #2 — Outlier removal pakai HARD CAP
before_outlier = len(df_flights_pd)
mask_active = df_flights_pd['cancelled'] == 0
mask_dep_ok = df_flights_pd['dep_delay'].between(-120, 1440)
df_flights_pd = df_flights_pd[~mask_active | mask_dep_ok]
print(f'  Outlier dep_delay dihapus: {before_outlier - len(df_flights_pd):,} baris')

# FIX #3 — arr_delay juga di-handle
before_arr = len(df_flights_pd)
mask_arr_ok = df_flights_pd['arr_delay'].between(-120, 1440)
df_flights_pd = df_flights_pd[~mask_active | mask_arr_ok]
print(f'  Outlier arr_delay dihapus: {before_arr - len(df_flights_pd):,} baris')

print(f'\n  Sesudah : {df_flights_pd.shape}')

# --- Cleaning df_airports ---
print('\n' + '=' * 55)
print(' CLEANING — df_airports')
print('=' * 55)
print(f'  Sebelum : {df_airports.shape}')

df_airports_pd = df_airports.to_pandas()
df_airports_pd.columns = [c.lower().strip() for c in df_airports_pd.columns]
df_airports_pd.drop_duplicates(subset=['iata'], inplace=True)
df_airports_pd.dropna(subset=['iata'], inplace=True)
df_airports_pd = df_airports_pd[df_airports_pd['iata'].str.strip() != '']

print(f'  Sesudah : {df_airports_pd.shape}')

duration = round(time.time() - start_time, 2)

write_custom_log('ETL', 'Transform_Cleaning', 'SUCCESS',
                 'Data cleaning selesai.',
                 {
                     'Flights Shape Sebelum': str(df_flights.shape),
                     'Flights Shape Sesudah': str(df_flights_pd.shape),
                     'Duplikat Dihapus'     : f'{dup_removed:,}',
                     'Airports Shape Sesudah': str(df_airports_pd.shape),
                     'Outlier Method'       : 'Hard Cap dep_delay [-120, 1440]',
                     'Waktu Eksekusi'       : f'{duration} detik'
                 })

print('\n✅ Data Cleaning selesai')

 CLEANING — df_flights
  Sebelum : (150000, 32)
  Duplikat dihapus  : 13 baris
  Null kunci dihapus: 0 baris
  Tanggal invalid   : 0 baris
  fillna dep_delay: median=-2.0 (hanya non-cancelled)
  fillna arr_delay: median=-7.0 (hanya non-cancelled)
  Outlier dep_delay dihapus: 9 baris
  Outlier arr_delay dihapus: 0 baris

  Sesudah : (149978, 32)

 CLEANING — df_airports
  Sebelum : (6393, 13)
  Sesudah : (6372, 13)

✅ Data Cleaning selesai


/tmp/ipykernel_451/3108418197.py:68: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_flights_pd = df_flights_pd[~mask_active | mask_arr_ok]


### CELL 7 — Standarisasi & Normalisasi

In [ ]:
# ============================================================
# STANDARISASI
# FIX #4 : normalisasi dilakukan SETELAH outlier removal
# FIX #5 : fillna(0) sebelum scaler dihapus — NaN dari cancelled
#           tidak dinormalisasi
# ============================================================
start_time = time.time()

# Seragamkan nama kolom snake_case
df_flights_pd.columns  = [c.lower().replace(' ', '_') for c in df_flights_pd.columns]
df_airports_pd.columns = [c.lower().replace(' ', '_') for c in df_airports_pd.columns]

# Normalisasi Min-Max pada dep_delay & arr_delay
scaler = MinMaxScaler()
norm_cols_use = [c for c in ['dep_delay', 'arr_delay']
                 if c in df_flights_pd.columns]

# Fit hanya pada baris non-cancelled yang punya nilai lengkap
mask_valid_norm = (
    (df_flights_pd['cancelled'] == 0) &
    df_flights_pd[norm_cols_use].notna().all(axis=1)
)
scaler.fit(df_flights_pd.loc[mask_valid_norm, norm_cols_use])

# Transform hanya baris valid, baris lain tetap NaN
df_flights_pd.loc[mask_valid_norm,
    [c + '_norm' for c in norm_cols_use]] = scaler.transform(
    df_flights_pd.loc[mask_valid_norm, norm_cols_use])

for c in norm_cols_use:
    if c + '_norm' not in df_flights_pd.columns:
        df_flights_pd[c + '_norm'] = np.nan

# Encoding kolom kategorikal
if 'cancelled' in df_flights_pd.columns:
    df_flights_pd['cancelled_enc'] = df_flights_pd['cancelled'].astype(int)

# Pastikan tipe data konsisten
for col in ['dep_delay', 'arr_delay', 'air_time', 'distance']:
    if col in df_flights_pd.columns:
        df_flights_pd[col] = pd.to_numeric(df_flights_pd[col], errors='coerce')

duration = round(time.time() - start_time, 2)

write_custom_log('ETL', 'Transform_Standardize', 'SUCCESS',
                 'Standarisasi dan normalisasi selesai.',
                 {
                     'Kolom Dinormalisasi': str([c + '_norm' for c in norm_cols_use]),
                     'Kolom Di-encode'    : 'cancelled_enc',
                     'Scaler'             : 'MinMaxScaler (fit hanya non-cancelled)',
                     'Waktu Eksekusi'     : f'{duration} detik'
                 })

print('✅ Standarisasi selesai')
print(f'   Kolom df_flights  : {list(df_flights_pd.columns)}')
print(f'   Kolom df_airports : {list(df_airports_pd.columns)}')
print(f'\n   Range dep_delay_norm min={df_flights_pd["dep_delay_norm"].min():.3f}, '
      f'max={df_flights_pd["dep_delay_norm"].max():.3f}')

✅ Standarisasi selesai
   Kolom df_flights  : ['fl_date', 'airline', 'airline_dot', 'airline_code', 'dot_code', 'fl_number', 'origin', 'origin_city', 'dest', 'dest_city', 'crs_dep_time', 'dep_time', 'dep_delay', 'taxi_out', 'wheels_off', 'wheels_on', 'taxi_in', 'crs_arr_time', 'arr_time', 'arr_delay', 'cancelled', 'cancellation_code', 'diverted', 'crs_elapsed_time', 'elapsed_time', 'air_time', 'distance', 'delay_due_carrier', 'delay_due_weather', 'delay_due_nas', 'delay_due_security', 'delay_due_late_aircraft', 'dep_delay_norm', 'arr_delay_norm', 'cancelled_enc']
   Kolom df_airports : ['airportname', 'iata', 'icao', 'timezone', 'city_name', 'city_iata', 'utc_offset_hours', 'utc_offset_seconds', 'country_codea2', 'country_codea3', 'country_name', 'geopointlat', 'geopointlong']

   Range dep_delay_norm min=0.000, max=1.000


### CELL 8 — Enrichment & Feature Engineering (5 fitur baru)

In [ ]:
# ============================================================
# ENRICHMENT & FEATURE ENGINEERING
# FIX #6 : kolom airports_pd sudah snake_case setelah standarisasi
# FIX #7 : delay_status diberi label 'cancelled' untuk cancelled flights
# ============================================================
start_time = time.time()

print('=== FEATURE ENGINEERING ===')

# Nama kolom airports setelah standarisasi snake_case
airport_cols = [c for c in ['iata', 'airportname', 'city_name', 'country_name',
                             'timezone', 'geopointlat', 'geopointlong']
                if c in df_airports_pd.columns]
df_ap = df_airports_pd[airport_cols].copy()

# Merge 1: bandara ASAL
df_merged = pd.merge(df_flights_pd, df_ap,
                     left_on='origin', right_on='iata', how='left')
rename_origin = {c: f'origin_{c}' for c in airport_cols if c != 'iata'}
df_merged.rename(columns=rename_origin, inplace=True)
df_merged.drop(columns=['iata'], inplace=True, errors='ignore')

# Merge 2: bandara TUJUAN
df_merged = pd.merge(df_merged, df_ap,
                     left_on='dest', right_on='iata', how='left')
rename_dest = {c: f'dest_{c}' for c in airport_cols if c != 'iata'}
df_merged.rename(columns=rename_dest, inplace=True)
df_merged.drop(columns=['iata'], inplace=True, errors='ignore')

df_merged.drop(columns=['origin_city', 'dest_city'], inplace=True, errors='ignore')

df_merged.rename(columns={
    'origin_airportname' : 'origin_airport_name',
    'dest_airportname'   : 'dest_airport_name',
    'origin_city_name'   : 'origin_city',
    'dest_city_name'     : 'dest_city',
    'origin_country_name': 'origin_country',
    'dest_country_name'  : 'dest_country'
}, inplace=True)

# Hapus kolom duplikat
df_merged = df_merged.loc[:, ~df_merged.columns.duplicated(keep='first')]

print(f'✅ Merge selesai: {df_merged.shape[0]:,} baris | {df_merged.shape[1]} kolom')

# ---- 5 Fitur Baru ----
# Fitur 1: flight_year
df_merged['flight_year']  = df_merged['fl_date'].dt.year
print('  [F1] flight_year')

# Fitur 2: flight_month
df_merged['flight_month'] = df_merged['fl_date'].dt.month
print('  [F2] flight_month')

# Fitur 3: day_of_week
df_merged['day_of_week']  = df_merged['fl_date'].dt.dayofweek
print('  [F3] day_of_week')

# Fitur 4: delay_status — label 'cancelled' untuk penerbangan cancelled
def assign_delay_status(row):
    if row.get('cancelled', 0) == 1:
        return 'cancelled'
    d = row.get('dep_delay', np.nan)
    if pd.isna(d):
        return 'unknown'
    if d <= 0:
        return 'on_time'
    elif d <= 15:
        return 'minor_delay'
    elif d <= 60:
        return 'moderate_delay'
    else:
        return 'severe_delay'

df_merged['delay_status'] = df_merged.apply(assign_delay_status, axis=1)
print('  [F4] delay_status')

# Fitur 5: is_weekend
df_merged['is_weekend'] = (df_merged['day_of_week'] >= 5).astype(int)
print('  [F5] is_weekend')

duration = round(time.time() - start_time, 2)

write_custom_log('ETL', 'Transform_FeatureEngineering', 'SUCCESS',
                 'Feature engineering & enrichment selesai.',
                 {
                     'Fitur Baru'    : 'flight_year, flight_month, day_of_week, delay_status, is_weekend',
                     'Shape Merged'  : str(df_merged.shape),
                     'Merge Sumber'  : 'flights JOIN airports (origin) JOIN airports (dest)',
                     'Waktu Eksekusi': f'{duration} detik'
                 })

print(f'\n✅ Feature Engineering selesai — 5 fitur baru ({duration}s)')
print(f'   Final: {df_merged.shape[0]:,} baris | {df_merged.shape[1]} kolom')
print(f'\n   Distribusi delay_status:')
print(df_merged['delay_status'].value_counts().to_string())

=== FEATURE ENGINEERING ===
✅ Merge selesai: 149,978 baris | 45 kolom
  [F1] flight_year
  [F2] flight_month
  [F3] day_of_week
  [F4] delay_status
  [F5] is_weekend

✅ Feature Engineering selesai — 5 fitur baru (1.57s)
   Final: 149,978 baris | 50 kolom

   Distribusi delay_status:
delay_status
on_time           96379
minor_delay       24114
moderate_delay    16584
severe_delay       8963
cancelled          3938


### CELL 9 — Validasi Kualitas Data (6 aturan wajib)

In [ ]:
# ============================================================
# VALIDASI KUALITAS DATA (6 aturan wajib)
# FIX #8 : Referential integrity cek persentase match (informatif)
# FIX #9 : Validasi distribusi dep_delay — mean & std masuk akal
# ============================================================
start_time = time.time()

hasil    = {}
perbaikan = []

# 1. Uniqueness check
dup_cols = [c for c in ['fl_date', 'airline', 'origin', 'dest', 'dep_time']
            if c in df_merged.columns]
n_dup = df_merged.duplicated(subset=dup_cols, keep=False).sum()
hasil['1. Uniqueness'] = 'PASS ✅' if n_dup == 0 else f'FAIL ❌ ({n_dup} duplikat)'

# 2. Null check kolom kritis
nulls = df_merged[['airline', 'origin', 'dest', 'fl_date']].isnull().sum().sum()
hasil['2. Null Check'] = 'PASS ✅' if nulls == 0 else f'FAIL ❌ ({nulls} nulls pada kolom kunci)'

# 3. Range check dep_delay (hard cap -120 s.d. 1440)
if 'dep_delay' in df_merged.columns:
    mask_nc   = df_merged['cancelled'] == 0
    extremes  = ((df_merged.loc[mask_nc, 'dep_delay'] < -120) |
                 (df_merged.loc[mask_nc, 'dep_delay'] > 1440)).sum()
    hasil['3. Range Check'] = ('PASS ✅' if extremes == 0
                               else f'FAIL ❌ ({extremes} nilai di luar [-120, 1440])')
else:
    hasil['3. Range Check'] = 'SKIP'

# 4. Datatype consistency
is_dt = pd.api.types.is_datetime64_any_dtype(df_merged['fl_date'])
hasil['4. Datatype Consistency'] = 'PASS ✅' if is_dt else 'FAIL ❌ (fl_date bukan datetime)'

# 5. Referential integrity — cek persentase match ke airports
ref_col = next((c for c in ['origin_airport_name', 'origin_city']
                if c in df_merged.columns), None)
if ref_col:
    no_match   = df_merged[ref_col].isnull().sum()
    total_rows = len(df_merged)
    match_pct  = round((total_rows - no_match) / total_rows * 100, 1)
    if no_match > 0:
        hasil['5. Referential Integrity'] = f'WARN ⚠️ ({no_match} origin tidak match | match rate: {match_pct}%)'
        perbaikan.append(f'  Ref. integrity: {no_match} origin tidak ada di airports.csv (bukan error fatal)')
    else:
        hasil['5. Referential Integrity'] = 'PASS ✅ (100% match)'
else:
    hasil['5. Referential Integrity'] = 'SKIP'

# 6. Distribusi dep_delay — mean & std harus masuk akal
if 'dep_delay' in df_merged.columns:
    mask_nc    = df_merged['cancelled'] == 0
    delay_mean = df_merged.loc[mask_nc, 'dep_delay'].mean()
    delay_std  = df_merged.loc[mask_nc, 'dep_delay'].std()
    dist_ok    = (delay_std > 5) and (-10 < delay_mean < 60)
    hasil['6. Distribusi'] = (
        f'PASS ✅ (mean={delay_mean:.2f} mnt, std={delay_std:.2f})'
        if dist_ok
        else f'WARN ⚠️ (mean={delay_mean:.2f} mnt, std={delay_std:.2f}) — distribusi mencurigakan')
else:
    hasil['6. Distribusi'] = 'SKIP'

duration = round(time.time() - start_time, 2)

print('=' * 55)
print('  HASIL VALIDASI KUALITAS DATA')
print('=' * 55)
for k, v in hasil.items():
    print(f'  {k}: {v}')
print('=' * 55)
print(f'\n  Total baris : {df_merged.shape[0]:,}')
print(f'  Total kolom : {df_merged.shape[1]}')

if perbaikan:
    print('\n  Catatan perbaikan:')
    for p in perbaikan:
        print(p)

# Simpan ke cleaned
clean_path = os.path.join(CLEAN_DIR, 'cleaned_flights_merged.csv')
df_merged.to_csv(clean_path, index=False)

write_custom_log('ETL', 'Transform_Validation', 'SUCCESS',
                 'Validasi 6 aturan kualitas data selesai.',
                 {
                     'Final Shape'           : str(df_merged.shape),
                     'Uniqueness Check'      : hasil['1. Uniqueness'],
                     'Null Check Kritis'     : hasil['2. Null Check'],
                     'Range Check Delay'     : hasil['3. Range Check'],
                     'Datatype Consistency'  : hasil['4. Datatype Consistency'],
                     'Referential Integrity' : hasil['5. Referential Integrity'],
                     'Distribusi Data'       : hasil['6. Distribusi'],
                     'Path Cleaned'          : clean_path,
                     'Waktu Eksekusi'        : f'{duration} detik'
                 })

print(f'\n✅ Validasi selesai ({duration}s) → {clean_path}')

  HASIL VALIDASI KUALITAS DATA
  1. Uniqueness: PASS ✅
  2. Null Check: PASS ✅
  3. Range Check: PASS ✅
  4. Datatype Consistency: PASS ✅
  5. Referential Integrity: WARN ⚠️ (177 origin tidak match | match rate: 99.9%)
  6. Distribusi: PASS ✅ (mean=10.09 mnt, std=48.54)

  Total baris : 149,978
  Total kolom : 50

  Catatan perbaikan:
  Ref. integrity: 177 origin tidak ada di airports.csv (bukan error fatal)

✅ Validasi selesai (0.07s) → /content/drive/MyDrive/TUBES BIG DATA/cleaned/cleaned_flights_merged.csv


## 🔴 LOAD
### CELL 10 — Buat Star Schema

In [ ]:
# ============================================================
# STAR SCHEMA
# FIX #10: dim_airline diisi nama maskapai LENGKAP
# FIX #11: fact_flights hanya filter origin (wajib), dest boleh NULL
# FIX #12: dim_date di-sort ascending
# ============================================================
start_time = time.time()
print('Membuat Star Schema DataFrames...\n')

# --- dim_airline ---
AIRLINE_NAMES = {
    'WN': 'Southwest Airlines Co.',    'DL': 'Delta Air Lines Inc.',
    'AA': 'American Airlines Inc.',    'OO': 'SkyWest Airlines Inc.',
    'UA': 'United Air Lines Inc.',     'YX': 'Republic Airline',
    'MQ': 'Envoy Air',                 'OH': 'Endeavor Air Inc.',
    'B6': 'JetBlue Airways',           'YV': 'Mesa Airlines Inc.',
    'G4': 'Allegiant Air',             'NK': 'Spirit Air Lines',
    'F9': 'Frontier Airlines Inc.',    'AS': 'Alaska Airlines Inc.',
    'HA': 'Hawaiian Airlines Inc.',    'VX': 'Virgin America',
    'EV': 'ExpressJet Airlines',       '9E': 'Endeavor Air Inc.',
    'CP': 'Compass Airlines',          'ZW': 'Air Wisconsin Airlines',
    'PT': 'Piedmont Airlines',         'QX': 'Horizon Air',
}
df_dim_airline = df_merged[['airline']].drop_duplicates().copy()
df_dim_airline.columns = ['airline_code']
df_dim_airline['airline_name'] = df_dim_airline['airline_code'].map(AIRLINE_NAMES)
df_dim_airline['airline_name'] = df_dim_airline['airline_name'].fillna(
    df_dim_airline['airline_code'])
df_dim_airline = df_dim_airline.reset_index(drop=True)
print(f'✅ dim_airline  : {len(df_dim_airline):,} baris')

# --- dim_airport ---
ap_cols_map = {
    'iata'        : 'iata_code',
    'airportname' : 'airport_name',
    'city_name'   : 'city',
    'country_name': 'country',
    'geopointlat' : 'latitude',
    'geopointlong': 'longitude',
    'timezone'    : 'timezone'
}
available = [c for c in ap_cols_map.keys() if c in df_airports_pd.columns]
df_dim_airport = df_airports_pd[available].copy()
df_dim_airport.rename(columns=ap_cols_map, inplace=True)
df_dim_airport.drop_duplicates(subset=['iata_code'], inplace=True)
df_dim_airport = df_dim_airport.reset_index(drop=True)
print(f'✅ dim_airport  : {len(df_dim_airport):,} baris')

# --- dim_date --- (FIX #12: sort ascending)
df_date = df_merged[['fl_date', 'flight_year', 'flight_month',
                      'day_of_week', 'is_weekend']].drop_duplicates().copy()
df_date['date_id'] = df_date['fl_date'].dt.strftime('%Y%m%d')
df_date['day']     = df_date['fl_date'].dt.day
df_date['quarter'] = df_date['fl_date'].dt.quarter
df_date.rename(columns={'flight_year': 'year', 'flight_month': 'month'}, inplace=True)
df_date = (df_date[['date_id', 'fl_date', 'year', 'month', 'day',
                    'day_of_week', 'is_weekend', 'quarter']]
           .drop_duplicates(subset=['date_id'])
           .sort_values('fl_date')
           .reset_index(drop=True))
print(f'✅ dim_date     : {len(df_date):,} baris')
print(f'   Rentang: {df_date["fl_date"].min().date()} s.d. {df_date["fl_date"].max().date()}')

# --- fact_flights --- (FIX #11: hanya filter origin)
fact_cols = ['airline', 'origin', 'dest', 'fl_date',
             'dep_delay', 'arr_delay', 'distance', 'cancelled']
opt_cols  = ['air_time', 'dep_delay_norm', 'arr_delay_norm',
             'delay_status', 'flight_month', 'cancelled_enc',
             'origin_airport_name', 'dest_airport_name',
             'origin_city', 'dest_city', 'origin_country', 'dest_country']
use_cols  = [c for c in fact_cols + opt_cols if c in df_merged.columns]

df_fact = df_merged[use_cols].copy()
df_fact['date_id'] = df_fact['fl_date'].dt.strftime('%Y%m%d')
df_fact.rename(columns={
    'airline': 'airline_code',
    'origin' : 'origin_code',
    'dest'   : 'dest_code'
}, inplace=True)
df_fact.drop(columns=['fl_date'], inplace=True)

valid_iatas = set(df_dim_airport['iata_code'])
before_ri   = len(df_fact)
df_fact     = df_fact[df_fact['origin_code'].isin(valid_iatas)]
dropped_ri  = before_ri - len(df_fact)
print(f'\n✅ fact_flights : {len(df_fact):,} baris')
print(f'   (dibuang karena origin tidak di dim_airport: {dropped_ri:,} baris)')

duration = round(time.time() - start_time, 2)

write_custom_log('ETL', 'Load_StarSchema_Build', 'SUCCESS',
                 'Star Schema berhasil dibuat.',
                 {
                     'fact_flights'   : f'{len(df_fact):,} baris',
                     'dim_airline'    : f'{len(df_dim_airline):,} baris',
                     'dim_airport'    : f'{len(df_dim_airport):,} baris',
                     'dim_date'       : f'{len(df_date):,} baris',
                     'Waktu Eksekusi' : f'{duration} detik'
                 })

Membuat Star Schema DataFrames...

✅ dim_airline  : 18 baris
✅ dim_airport  : 6,372 baris
✅ dim_date     : 1,704 baris
   Rentang: 2019-01-01 s.d. 2023-08-31

✅ fact_flights : 149,801 baris
   (dibuang karena origin tidak di dim_airport: 177 baris)


### CELL 11 — 8 Query SQL Analitik

In [ ]:
# ============================================================
# 8 QUERY SQL ANALITIK
# FIX #13: Query 1 — avg delay exclude cancelled
# FIX #14: Query 5 — on-time rate dihitung benar
# FIX #15: Query 7 — total_cancelled cast ke int
# ============================================================
import sqlite3

# Simpan ke SQLite untuk query lokal
sqlite_path = os.path.join(WH_DIR, 'flight_warehouse.db')
conn = sqlite3.connect(sqlite_path)

def prepare_sqlite(df):
    df = df.copy()
    for col in df.columns:
        if str(df[col].dtype) == 'category':
            df[col] = df[col].astype(str)
        elif 'datetime' in str(df[col].dtype):
            df[col] = df[col].dt.strftime('%Y-%m-%d')
    return df

for nama, df_t in [('fact_flights', df_fact), ('dim_airline', df_dim_airline),
                    ('dim_airport', df_dim_airport), ('dim_date', df_date)]:
    prepare_sqlite(df_t).to_sql(nama, conn, if_exists='replace', index=False)

print('8 Query Analitik\n')

# Query 1: Top 10 Maskapai delay terlama (exclude cancelled)
print('=' * 55)
print(' 1. Top 10 Maskapai Delay Terlama (exclude cancelled)')
print('=' * 55)
q1 = (df_fact[df_fact['cancelled'] == 0]
      .merge(df_dim_airline, on='airline_code', how='left')
      .groupby('airline_name')
      .agg(avg_dep_delay    =('dep_delay', 'mean'),
           total_penerbangan=('dep_delay', 'count'))
      .round(2)
      .sort_values('avg_dep_delay', ascending=False)
      .head(10))
print(q1.to_string())

# Query 2: Tren Penerbangan per Tahun
print('\n' + '=' * 55)
print(' 2. Tren Penerbangan per Tahun')
print('=' * 55)
df_f2 = df_fact.merge(df_date[['date_id', 'year']], on='date_id', how='left')
q2 = (df_f2.groupby('year')
      .agg(total_flights   =('dep_delay', 'count'),
           avg_delay_menit =('dep_delay', 'mean'),
           total_cancelled =('cancelled', 'sum'))
      .round(2).sort_values('year'))
print(q2.to_string())

# Query 3: Bandara Asal Tersibuk
print('\n' + '=' * 55)
print(' 3. Bandara Asal Tersibuk')
print('=' * 55)
q3 = (df_fact
      .merge(df_dim_airport[['iata_code', 'airport_name', 'city', 'country']],
             left_on='origin_code', right_on='iata_code', how='left')
      .groupby(['airport_name', 'city', 'country'])
      .size().reset_index(name='total_flights')
      .sort_values('total_flights', ascending=False).head(10))
print(q3.to_string(index=False))

# Query 4: Weekend vs Weekday Delay
print('\n' + '=' * 55)
print(' 4. Weekend vs Weekday Delay (exclude cancelled)')
print('=' * 55)
df_f4 = (df_fact[df_fact['cancelled'] == 0]
         .merge(df_date[['date_id', 'is_weekend']], on='date_id', how='left'))
q4 = (df_f4.groupby('is_weekend')
      .agg(avg_delay=('dep_delay', 'mean'), total=('dep_delay', 'count'))
      .round(2))
q4.index = q4.index.map({0: 'Weekday', 1: 'Weekend'})
print(q4.to_string())

# Query 5: Distribusi Status Delay + On-Time Rate (FIX #14)
print('\n' + '=' * 55)
print(' 5. Distribusi Status Delay & On-Time Rate')
print('=' * 55)
q5 = (df_fact.groupby('delay_status').size()
      .reset_index(name='jumlah').sort_values('jumlah', ascending=False))
q5['persen'] = (q5['jumlah'] / q5['jumlah'].sum() * 100).round(2)
print(q5.to_string(index=False))

total_active  = (df_fact['cancelled'] == 0).sum()
total_on_time = (df_fact['delay_status'] == 'on_time').sum()
on_time_rate  = round(total_on_time / total_active * 100, 2) if total_active > 0 else 0
print(f'\n  On-Time Rate (exclude cancelled): {on_time_rate}%')
print(f'  Total penerbangan aktif         : {total_active:,}')
print(f'  Total on-time                   : {total_on_time:,}')

# Query 6: Rute Terpopuler
print('\n' + '=' * 55)
print(' 6. Rute Terpopuler (Origin → Dest)')
print('=' * 55)
q6 = (df_fact.groupby(['origin_code', 'dest_code'])
      .agg(total=('dep_delay', 'count'), avg_delay=('dep_delay', 'mean'))
      .round(2).sort_values('total', ascending=False).head(10))
print(q6.to_string())

# Query 7: Pembatalan per Bulan (FIX #15: cast int)
print('\n' + '=' * 55)
print(' 7. Pembatalan per Bulan')
print('=' * 55)
df_f7 = df_fact.merge(df_date[['date_id', 'year', 'month']], on='date_id', how='left')
q7 = (df_f7.groupby(['year', 'month'])
      .agg(total_cancelled=('cancelled', 'sum'), total_flights=('cancelled', 'count'))
      .sort_values(['year', 'month']))
q7['total_cancelled'] = q7['total_cancelled'].astype(int)  # FIX: cast ke int
q7['pct_cancelled']   = (q7['total_cancelled'] / q7['total_flights'] * 100).round(2)
print(q7.to_string())

# Query 8: Jarak Penerbangan vs Delay
print('\n' + '=' * 55)
print(' 8. Jarak Penerbangan vs Delay (exclude cancelled)')
print('=' * 55)
df_f8 = df_fact[df_fact['cancelled'] == 0].copy()
df_f8['distance_group'] = pd.cut(
    df_f8['distance'],
    bins=[0, 500, 1500, float('inf')],
    labels=['Short (<500mi)', 'Medium (500-1500mi)', 'Long (>1500mi)'])
q8 = (df_f8.groupby('distance_group', observed=True)
      .agg(avg_dep_delay=('dep_delay', 'mean'), total=('dep_delay', 'count'))
      .round(2).sort_values('avg_dep_delay', ascending=False))
print(q8.to_string())

conn.close()

write_custom_log('ETL', 'SQL_Analytics', 'SUCCESS',
                 '8 query analitik berhasil dijalankan.',
                 {
                     'Query Dijalankan': '8 query (avg delay, tren, bandara, weekend, distribusi, rute, pembatalan, jarak)',
                     'SQLite Path'     : sqlite_path
                 })

print('\n✅ 8 Query SQL Analitik selesai!')

8 Query Analitik

 1. Top 10 Maskapai Delay Terlama (exclude cancelled)
                                    avg_dep_delay  total_penerbangan
airline_name                                                        
JetBlue Airways                             18.71               5414
Frontier Airlines Inc.                      15.19               3245
Allegiant Air                               13.85               2497
Spirit Air Lines                            13.69               4783
American Airlines Inc.                      12.52              18460
Mesa Airlines Inc.                          12.07               3083
ExpressJet Airlines LLC d/b/a aha!          11.13                908
United Air Lines Inc.                       11.06              12363
Southwest Airlines Co.                      10.85              27419
SkyWest Airlines Inc.                        9.62              16962

 2. Tren Penerbangan per Tahun
      total_flights  avg_delay_menit  total_cancelled
year          

### CELL 12 — Export ke BigQuery

In [ ]:
!pip install pandas-gbq google-cloud-bigquery -q

from google.colab import auth
import pandas_gbq
from google.cloud import bigquery

print('Autentikasi Google...')
auth.authenticate_user()
print('✅ Autentikasi berhasil\n')

client      = bigquery.Client(project=GCP_PROJECT)
dataset_ref = bigquery.Dataset(f'{GCP_PROJECT}.{BQ_DATASET}')
dataset_ref.location = 'US'
client.create_dataset(dataset_ref, exists_ok=True)
print(f'✅ Dataset BigQuery siap : {GCP_PROJECT}.{BQ_DATASET}\n')

Autentikasi Google...
✅ Autentikasi berhasil

✅ Dataset BigQuery siap : tubes-bigdata-498212.etl_flight_analytics



In [ ]:
# ============================================================
# FUNGSI PERSIAPAN DataFrame SEBELUM UPLOAD
# FIX #16: handle 'category' dtype dengan .where() yang aman
# FIX #17: delay_status string biasa, tidak perlu konversi khusus
# ============================================================

def prepare_df(df):
    df = df.copy().reset_index(drop=True)
    df = df.loc[:, ~df.columns.duplicated(keep='first')]

    for col in df.columns:
        dtype_str = str(df[col].dtype)

        if dtype_str == 'category':
            df[col] = df[col].astype(str)
            df[col] = df[col].where(df[col] != 'nan', other=np.nan)
        elif 'datetime' in dtype_str:
            df[col] = df[col].dt.strftime('%Y-%m-%d').where(df[col].notna(), other=None)
        elif dtype_str == 'object':
            df[col] = df[col].where(df[col].notna(), other=None)
    return df

# Siapkan etl_transformed (tabel utama untuk dashboard)
df_etl_transformed = df_merged.copy()
df_etl_transformed = df_etl_transformed.loc[
    :, ~df_etl_transformed.columns.duplicated(keep='first')]
if 'fl_date' in df_etl_transformed.columns:
    df_etl_transformed['fl_date'] = (
        df_etl_transformed['fl_date'].dt.strftime('%Y-%m-%d'))

upload_list = {
    'etl_transformed': df_etl_transformed,
    'fact_flights'   : df_fact.copy(),
    'dim_airline'    : df_dim_airline.copy(),
    'dim_airport'    : df_dim_airport.copy(),
    'dim_date'       : df_date.copy(),
}

print('Upload ke BigQuery...\n')
sukses = []
gagal  = []

for nama, df_up_raw in upload_list.items():
    start_load = time.time()
    try:
        df_up = prepare_df(df_up_raw)
        pandas_gbq.to_gbq(
            df_up,
            destination_table=f'{BQ_DATASET}.{nama}',
            project_id=GCP_PROJECT,
            if_exists='replace',
            progress_bar=True
        )
        load_duration = round(time.time() - start_load, 2)
        sukses.append(nama)

        write_custom_log('ETL', f'Load_Table_{nama}', 'SUCCESS',
                         f'Tabel {nama} dimuat ke BigQuery.',
                         {
                             'Target Table Warehouse'     : f'{BQ_DATASET}.{nama}',
                             'Jumlah Baris Terunggah'    : f'{len(df_up):,}',
                             'Jumlah Kolom'              : df_up.shape[1],
                             'Waktu Pengiriman (Load Time)': f'{load_duration} detik'
                         })

        print(f'  ✅ {nama:20s} → BigQuery ({len(df_up):,} baris | {df_up.shape[1]} kolom | {load_duration}s)')

    except Exception as e:
        load_duration = round(time.time() - start_load, 2)
        gagal.append(nama)
        write_custom_log('ETL', f'Load_Table_{nama}', 'FAILED',
                         f'Gagal muat tabel. Error: {str(e)}')
        print(f'  ❌ {nama:20s} gagal: {e}')

total_baris = sum(len(upload_list[n]) for n in sukses)

write_custom_log('ETL', 'Load_Summary', 'SUCCESS',
                 'Semua tabel selesai diproses.',
                 {
                     'Tabel Berhasil'  : str(sukses),
                     'Tabel Gagal'     : str(gagal) if gagal else 'tidak ada',
                     'Total Baris'     : f'{total_baris:,}',
                     'Dataset BigQuery': f'{GCP_PROJECT}.{BQ_DATASET}'
                 })

print(f"""
╔══════════════════════════════════════════════════╗
  ✅ ETL → BigQuery selesai!
  Berhasil     : {len(sukses)} tabel {sukses}
  Gagal        : {len(gagal)} tabel {gagal if gagal else '-'}
  Total baris  : {total_baris:,}
  Dataset      : {GCP_PROJECT}.{BQ_DATASET}
╚══════════════════════════════════════════════════╝
"""
)

Upload ke BigQuery...



100%|██████████| 1/1 [00:00<00:00, 9597.95it/s]


  ✅ etl_transformed      → BigQuery (149,978 baris | 50 kolom | 22.57s)


100%|██████████| 1/1 [00:00<00:00, 11586.48it/s]


  ✅ fact_flights         → BigQuery (149,801 baris | 20 kolom | 5.6s)


...

  ✅ dim_airline          → BigQuery (18 baris | 2 kolom | 3.17s)


100%|██████████| 1/1 [00:00<00:00, 13888.42it/s]


  ✅ dim_airport          → BigQuery (6,372 baris | 7 kolom | 4.07s)


100%|██████████| 1/1 [00:00<00:00, 15592.21it/s]

  ✅ dim_date             → BigQuery (1,704 baris | 8 kolom | 3.01s)

╔══════════════════════════════════════════════════╗
  ✅ ETL → BigQuery selesai!
  Berhasil     : 5 tabel ['etl_transformed', 'fact_flights', 'dim_airline', 'dim_airport', 'dim_date']
  Gagal        : 0 tabel -
  Total baris  : 307,873
  Dataset      : tubes-bigdata-498212.etl_flight_analytics
╚══════════════════════════════════════════════════╝



## ✅ CELL 13 — Verifikasi Log & Ringkasan Akhir

In [ ]:
# ============================================================
# VERIFIKASI LOG — pastikan etl_log.txt tersimpan di Drive
# ============================================================
print('=== VERIFIKASI LOG ===')

if os.path.exists(log_path):
    size_kb = round(os.path.getsize(log_path) / 1024, 2)
    print(f'✅ etl_log.txt ditemukan: {size_kb} KB')
    print(f'   Path: {log_path}\n')
    print('--- ISI LOG (50 baris terakhir) ---')
    with open(log_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(''.join(lines[-50:]))
else:
    print('❌ etl_log.txt TIDAK ditemukan — cek BASE_DIR!')

# Tulis footer
with open(log_path, 'a', encoding='utf-8') as f:
    footer = (
        '\n' + '=' * 60 + '\n'
        f'Pipeline selesai pada: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
        '=== END OF ETL LOG ===\n'
    )
    f.write(footer)
    f.flush()

print('\n🎉 PIPELINE ETL SELESAI SEPENUHNYA!')
print(f'   Log       → {log_path}')
print(f'   Raw       → {RAW_DIR}')
print(f'   Cleaned   → {CLEAN_DIR}')
print(f'   Warehouse → {WH_DIR}')
print(f'   BigQuery  → {GCP_PROJECT}.{BQ_DATASET}')

=== VERIFIKASI LOG ===
✅ etl_log.txt ditemukan: 4.4 KB
   Path: /content/drive/MyDrive/TUBES BIG DATA/etl_log.txt

--- ISI LOG (50 baris terakhir) ---
   > Waktu Eksekusi: 1.57 detik
[2026-06-14 08:39:43] [ETL] [TRANSFORM_VALIDATION] [SUCCESS] - Validasi 6 aturan kualitas data selesai.
   > Final Shape: (149978, 50)
   > Uniqueness Check: PASS ✅
   > Null Check Kritis: PASS ✅
   > Range Check Delay: PASS ✅
   > Datatype Consistency: PASS ✅
   > Referential Integrity: WARN ⚠️ (177 origin tidak match | match rate: 99.9%)
   > Distribusi Data: PASS ✅ (mean=10.09 mnt, std=48.54)
   > Path Cleaned: /content/drive/MyDrive/TUBES BIG DATA/cleaned/cleaned_flights_merged.csv
   > Waktu Eksekusi: 0.07 detik
[2026-06-14 08:39:44] [ETL] [LOAD_STARSCHEMA_BUILD] [SUCCESS] - Star Schema berhasil dibuat.
   > fact_flights: 149,801 baris
   > dim_airline: 18 baris
   > dim_airport: 6,372 baris
   > dim_date: 1,704 baris
   > Waktu Eksekusi: 0.79 detik
[2026-06-14 08:39:46] [ETL] [SQL_ANALYTICS] [SUCCESS